# Zebrafish: GitHub setup + script walkthrough

Goal: capture *exactly what we did* to set up GitHub tracking for the BIOL550 group project zebrafish workspace, with safety checks to avoid re-doing work (re-cloning, re-adding submodules, etc.).

This notebook focuses on:
- How `group_project` is tracked (as a submodule)
- What the zebrafish scripts do and the main commands they run
- A cell-by-cell walkthrough of the `get_zebrafish_data_sra.py` script

Everything is organized under:
- `Semester5/BIOL550/group_project/zebrafish/`

## Outline

1. Setup + safety switches
2. Verify local repos (BIOL550 parent + `group_project` submodule)
3. Verify GitHub remotes (what points to where)
4. What we built for zebrafish (files + responsibilities)
5. Script walkthrough: `scripts/get_zebrafish_data_sra.py`
6. “Command cheat sheet” (copy/paste)

## Server deploy (like Lab 1)

The goal is for the server to have a working copy of the repo at a stable path (e.g., `/home/zebrafish`), so we can run the same scripts there.

Important notes:
- If you **do not have write permission** to `/home/zebrafish`, use `~/zebrafish` instead.
- If the server **does not have `rsync`**, prefer `git clone` (best) or `tar | ssh` as fallback.
- If `git@github.com:...` fails on the server (no SSH key), use the HTTPS remote.

In [ ]:
# Fill these in with your real values
SERVER = "USER@HOST"          # e.g., "pzg8794@sequoia.rit.edu"
SERVER_DEST = "/home/zebrafish"  # or "~/zebrafish" if you don't have permission

GROUP_PROJECT_REPO_SSH = "git@github.com:pzg8794/BIOL550-group_project.git"
GROUP_PROJECT_REPO_HTTPS = "https://github.com/pzg8794/BIOL550-group_project.git"

print('SERVER:', SERVER)
print('SERVER_DEST:', SERVER_DEST)
print('Repo (SSH):', GROUP_PROJECT_REPO_SSH)
print('Repo (HTTPS):', GROUP_PROJECT_REPO_HTTPS)

### Safety check: can we write to the destination?

Run this *from your laptop* in a terminal (or set `ALLOW_WRITE_ACTIONS=True` and run the next code cell).

In [ ]:
cmd = f"ssh {SERVER} 'ls -ld {SERVER_DEST} && touch {SERVER_DEST}/.write_test && rm -f {SERVER_DEST}/.write_test'"
print(cmd)

if ALLOW_WRITE_ACTIONS:
    print(sh(cmd).stdout)
else:
    print('Not executed (ALLOW_WRITE_ACTIONS=False).')

### Clone/pull the repo on the server (idempotent)

This will:
- If `{SERVER_DEST}/.git` exists: `git pull`
- Else: `git clone` into `{SERVER_DEST}`

It also falls back to HTTPS if SSH clone fails.

In [ ]:
remote_script = (
    "set -e; "
    "if [ -d '{dest}/.git' ]; then "
    "  cd '{dest}'; git remote -v; git pull --ff-only; "
    "elif [ -e '{dest}' ]; then "
    "  echo 'ERROR: {dest} exists but is not a git repo (no .git). Refusing to overwrite.'; exit 2; "
    "else "
    "  (git clone '{ssh}' '{dest}' || git clone '{https}' '{dest}'); "
    "fi; "
    "cd '{dest}'; git status -sb; ls -la zebrafish | head"
)
remote_script = remote_script.format(dest=SERVER_DEST, ssh=GROUP_PROJECT_REPO_SSH, https=GROUP_PROJECT_REPO_HTTPS)

cmd = f"ssh {SERVER} \"{remote_script}\""
print(cmd)

if ALLOW_WRITE_ACTIONS:
    print(sh(cmd).stdout)
else:
    print('Not executed (ALLOW_WRITE_ACTIONS=False).')

### Verify the notebooks exist on the server

After cloning, you should see:
- `zebrafish/zebrafish_sra_api_test_download.ipynb`
- `zebrafish/zebrafish_github_setup_and_script_walkthrough.ipynb`

In [ ]:
cmd = f"ssh {SERVER} 'ls -la {SERVER_DEST}/zebrafish/*.ipynb 2>/dev/null || true'"
print(cmd)

if ALLOW_WRITE_ACTIONS:
    print(sh(cmd).stdout)
else:
    print('Not executed (ALLOW_WRITE_ACTIONS=False).')

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
from dataclasses import dataclass
from pathlib import Path

# ---------------------------
# Safety switches
# ---------------------------
# Keep these False by default so we don't change repo state accidentally.
ALLOW_WRITE_ACTIONS = False   # set True only if you intend to run git add/commit/push, etc.

# ---------------------------
# Paths (repo-local)
# ---------------------------
# Assumes this notebook is opened from the repository workspace.
ZEBRAFISH_ROOT = Path('Semester5/BIOL550/group_project/zebrafish').resolve()
BIOL550_ROOT = Path('Semester5/BIOL550').resolve()
GROUP_PROJECT_ROOT = BIOL550_ROOT / 'group_project'

SCRIPT_GET_ZEBRAFISH = ZEBRAFISH_ROOT / 'scripts' / 'get_zebrafish_data_sra.py'
NOTEBOOK_SRA_DOWNLOAD = ZEBRAFISH_ROOT / 'zebrafish_sra_api_test_download.ipynb'

EXPECTED_GROUP_PROJECT_REMOTE = 'git@github.com:pzg8794/BIOL550-group_project.git'


def which(cmd: str) -> str | None:
    return shutil.which(cmd)


def sh(cmd: str, *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    """Run a shell command and return CompletedProcess.

    This prints the command first so the notebook acts like a work log.
    """
    print('$', cmd)
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=True,
        check=check,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )


def assert_exists(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(path)


print('BIOL550_ROOT:', BIOL550_ROOT)
print('GROUP_PROJECT_ROOT:', GROUP_PROJECT_ROOT)
print('ZEBRAFISH_ROOT:', ZEBRAFISH_ROOT)
print('Script exists:', SCRIPT_GET_ZEBRAFISH.exists())
print('Notebook exists:', NOTEBOOK_SRA_DOWNLOAD.exists())

## 1) Verify the BIOL550 repo is a git repo

This checks that the *parent* repo (`Semester5/BIOL550`) is tracked by Git.

In [ ]:
assert_exists(BIOL550_ROOT)
cp = sh('git rev-parse --is-inside-work-tree', cwd=BIOL550_ROOT)
print(cp.stdout.strip())
print(sh('git status --porcelain', cwd=BIOL550_ROOT).stdout[:2000])

## 2) Verify `group_project` is a submodule (safety: no re-adding)

We confirm `.gitmodules` exists and that `group_project` shows up in `git submodule status`.

In [ ]:
assert_exists(BIOL550_ROOT / '.gitmodules')
print((BIOL550_ROOT / '.gitmodules').read_text(encoding='utf-8')[:2000])

print(sh('git submodule status', cwd=BIOL550_ROOT).stdout)

# Check the submodule remote
assert_exists(GROUP_PROJECT_ROOT)
print(sh('git -C group_project remote -v', cwd=BIOL550_ROOT).stdout)

## 3) Verify GitHub remotes (what points to where)

This is the “don’t redo” safety: we just *inspect*.

- Parent repo: should point to your BIOL550 repo.
- Submodule repo: should point to `git@github.com:pzg8794/BIOL550-group_project.git`.

In [ ]:
print('Parent repo remotes:')
print(sh('git remote -v', cwd=BIOL550_ROOT).stdout)

print('Submodule remotes:')
print(sh('git remote -v', cwd=GROUP_PROJECT_ROOT).stdout)

# Optional assertion: warn if it doesn't match expected
remotes = sh('git remote -v', cwd=GROUP_PROJECT_ROOT).stdout
if EXPECTED_GROUP_PROJECT_REMOTE not in remotes:
    print('WARNING: group_project remote does not match expected:')
    print(' expected:', EXPECTED_GROUP_PROJECT_REMOTE)
else:
    print('OK: group_project remote matches expected.')

## Publish changes to GitHub (submodule)

For the server to receive the latest notebooks/scripts via `git clone`/`git pull`, you must **commit + push** changes in the `group_project` repo.

Safety: the next cells only *inspect* by default. Set `ALLOW_WRITE_ACTIONS=True` before running any `git commit` / `git push`.

In [ ]:
# Inspect what would be pushed (safe)
print(sh('git status -sb', cwd=GROUP_PROJECT_ROOT).stdout)
print(sh('git remote -v', cwd=GROUP_PROJECT_ROOT).stdout)

### Commit + push (only if you intend to)

Typical sequence:

```bash
cd Semester5/BIOL550/group_project
git add .
git commit -m "Update zebrafish workflow"
git push origin main
```

In [ ]:
cmds = [
    'git add .',
    'git commit -m "Update zebrafish workflow"',
    'git push origin main',
]
print('Commands to run inside the submodule:')
for c in cmds:
    print(' -', c)

if ALLOW_WRITE_ACTIONS:
    for c in cmds:
        print(sh(c, cwd=GROUP_PROJECT_ROOT).stdout)
else:
    print('Not executed (ALLOW_WRITE_ACTIONS=False).')

## 4) What we built for zebrafish (files + responsibilities)

This is the directory “map” so you can find things quickly.

In [ ]:
assert_exists(ZEBRAFISH_ROOT)

# list a focused subset (avoid printing huge trees)
for rel in [
    'README.md',
    'scripts/get_zebrafish_data_sra.py',
    'scripts/download_test_5_runs_fastq.sh',
    'scripts/download_fastq_sratoolkit.sh',
    'metadata/PRJNA1277581/runinfo.csv',
    'metadata/PRJNA1277581/runinfo.filtered.csv',
    'metadata/PRJNA1277581/runs.filtered.txt',
    'metadata/PRJNA1277581/runs.test5.smallest_sizeMB.txt',
    'notes/dataset.md',
    'zebrafish_sra_api_test_download.ipynb',
]:
    p = ZEBRAFISH_ROOT / rel
    print(rel, 'exists=' , p.exists())

## 5) Script walkthrough: `scripts/get_zebrafish_data_sra.py`

This script:
- Fetches **RunInfo CSV** for an SRA accession using the Entrez API (prefer `esearch`/`efetch` if installed).
- Filters runs by organism/layout/avgLength/spots.
- Writes **runinfo + SRR lists** into `metadata/<ACC>/`.

It does **not** download FASTQs; that stays in separate download scripts/notebooks.

In [ ]:
assert_exists(SCRIPT_GET_ZEBRAFISH)
text = SCRIPT_GET_ZEBRAFISH.read_text(encoding='utf-8')
print('--- first 80 lines ---')
print('\n'.join(text.splitlines()[:80]))

### 5.1 Main command the script runs (API metadata pull)

This is the core command (when Entrez Direct is available):

- `esearch -db sra -query <ACC> | efetch -format runinfo`

The script uses that to save `runinfo.csv`.

In [ ]:
# Extract the key pipeline from the script text (best-effort)
pattern = r"\[esearch,\s*'-db',\s*'sra',\s*'-query',\s*acc\]"
print('Contains esearch call:', bool(re.search(pattern, SCRIPT_GET_ZEBRAFISH.read_text(encoding='utf-8'))))
print('esearch in PATH:', which('esearch'))
print('efetch in PATH:', which('efetch'))

### 5.2 Script structure (functions)

This lists the functions defined in the script so you can navigate it quickly.

In [ ]:
import ast

tree = ast.parse(SCRIPT_GET_ZEBRAFISH.read_text(encoding='utf-8'))
funcs = [n.name for n in tree.body if isinstance(n, ast.FunctionDef)]
classes = [n.name for n in tree.body if isinstance(n, ast.ClassDef)]
print('Classes:', classes)
print('Functions:')
for f in funcs:
    print(' -', f)

### 5.3 Walkthrough (in plain English)

Key decisions in the script:

- Prefer Entrez Direct (`esearch`/`efetch`) when available because it’s reliable in class environments.
- Fallback to direct E-utilities HTTP calls if `esearch`/`efetch` aren’t installed.
- Keep outputs deterministic:
  - `runinfo.csv` (raw)
  - `runinfo.filtered.csv` (filtered)
  - `runs.all.txt` and `runs.filtered.txt`

Filtering defaults match our class validation rules:
- Organism: `Danio rerio`
- Strategy: `RNA-Seq`
- Layout: `PAIRED`
- `avgLength >= 150`
- `spots >= 40,000,000`

## 6) Command cheat sheet (copy/paste)

### Fetch RunInfo + SRR lists

```bash
python3 scripts/get_zebrafish_data_sra.py
```

### Download 5-run test subset (fast)

```bash
MAX_SPOTS=10000 bash scripts/download_test_5_runs_fastq.sh
```

### Download full FASTQs (requires SRA Toolkit; heavier)

```bash
bash scripts/download_fastq_sratoolkit.sh
```